# NSE Daily Stocks + NIFTY 50 Data Sync

Full historical notebook rebuilt from the original source. All original functionality is preserved; the post-July-2024 NSE stock download logic is patched in the dedicated repair cell below.


In [ ]:

The stock dataset keeps the existing schema:

`date, symbol, open, high, low, close, volume`

NIFTY 50 uses the same schema. Since an index itself is not a traded security, its `volume` is stored as null.

# ============================================================
# 1. SETUP
# ============================================================

!pip -q install pandas pyarrow requests duckdb tqdm

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import date, datetime, timedelta
import io
import time
import zipfile
import logging

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm
import duckdb

print("Environment ready.")

# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/quant")
DATA_DIR = BASE_DIR / "data"

# Existing stock data
PARQUET_DIR = DATA_DIR / "parquet"
RAW_DIR = DATA_DIR / "raw" / "nse_bhavcopy"
METADATA_DIR = DATA_DIR / "metadata"
MANIFEST_FILE = DATA_DIR / "download_manifest.csv"

# NIFTY 50 data
INDEX_DATA_DIR = DATA_DIR / "indices"
NIFTY50_DIR = INDEX_DATA_DIR / "nifty50"
NIFTY50_MANIFEST_FILE = INDEX_DATA_DIR / "nifty50_manifest.csv"

# Date range
START_DATE = date(2011, 1, 1)
END_DATE = None   # None = today

# NSE stock format transition
LEGACY_END_DATE = date(2024, 7, 5)
UDIFF_START_DATE = date(2024, 7, 8)

# Optional stock filter
# None = all stocks
# Example: {"RELIANCE", "TCS", "INFY"}
STOCKS = None

# Download behavior
REQUEST_TIMEOUT = 60
MAX_RETRIES = 5
BACKOFF_FACTOR = 1.5
SLEEP_BETWEEN_REQUESTS = 0.10

# Repair invalid/corrupt Parquet files
REPAIR_INVALID_FILES = True

# Keep downloaded raw ZIPs?
KEEP_RAW_ZIPS = False

EXPECTED_COLUMNS = [
    "date",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

# Create directories
for directory in [
    BASE_DIR,
    DATA_DIR,
    PARQUET_DIR,
    RAW_DIR,
    METADATA_DIR,
    INDEX_DATA_DIR,
    NIFTY50_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

if END_DATE is None:
    END_DATE = date.today()

assert START_DATE <= END_DATE

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("STOCK DATA:", PARQUET_DIR)
print("NIFTY50 DATA:", NIFTY50_DIR)
print("DATE RANGE:", START_DATE, "to", END_DATE)

# ============================================================
# 3. LOGGING + HTTP SESSION
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger("nse_sync")

session = requests.Session()

retry = Retry(
    total=MAX_RETRIES,
    connect=MAX_RETRIES,
    read=MAX_RETRIES,
    status=MAX_RETRIES,
    backoff_factor=BACKOFF_FACTOR,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
})

print("HTTP session configured.")

# ============================================================
# 4. COMMON DATE / PATH HELPERS
# ============================================================

def trading_day_candidates(start_date: date, end_date: date):
    current = start_date

    while current <= end_date:
        if current.weekday() < 5:
            yield current

        current += timedelta(days=1)


def parquet_path_for_day(day: date) -> Path:
    year_dir = PARQUET_DIR / f"year={day.year}"
    year_dir.mkdir(parents=True, exist_ok=True)

    return year_dir / f"nse_cm_{day:%Y%m%d}.parquet"


def nifty50_parquet_path_for_day(day: date) -> Path:
    year_dir = NIFTY50_DIR / f"year={day.year}"
    year_dir.mkdir(parents=True, exist_ok=True)

    return year_dir / f"nifty50_{day:%Y%m%d}.parquet"


def raw_zip_path_for_day(day: date) -> Path:
    return RAW_DIR / f"nse_cm_{day:%Y%m%d}.zip"


print("Path helpers ready.")

# ============================================================
# 5. NSE STOCK URL HELPERS
# ============================================================

def legacy_url(day: date) -> str:
    month = day.strftime("%b").upper()

    filename = (
        f"cm{day:%d}{month}{day:%Y}bhav.csv.zip"
    )

    return (
        "https://nsearchives.nseindia.com/content/"
        f"historical/EQUITIES/{day.year}/{month}/{filename}"
    )


def udiff_url(day: date) -> str:
    filename = (
        f"BhavCopy_NSE_CM_0_0_0_"
        f"{day:%Y%m%d}_F_0000.csv.zip"
    )

    return (
        "https://nsearchives.nseindia.com/content/cm/"
        f"{filename}"
    )


def url_for_day(day: date) -> str:
    if day <= LEGACY_END_DATE:
        return legacy_url(day)

    return udiff_url(day)


for d in [
    date(2011, 1, 3),
    date(2024, 7, 5),
    date(2024, 7, 8),
]:
    print(d, "->", url_for_day(d))

# ============================================================
# 6. STOCK DATA NORMALIZATION
# ============================================================

def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace("/", "_")
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}

    for col in df.columns:
        n = normalize_column_name(col)

        aliases = {
            "tradingsymbol": "symbol",
            "symbol": "symbol",

            "timestamp": "date",
            "trade_date": "date",
            "date": "date",

            "open_price": "open",
            "open": "open",

            "high_price": "high",
            "high": "high",

            "low_price": "low",
            "low": "low",

            "close_price": "close",
            "close": "close",

            "last_price": "close",
            "ltp": "close",

            "tottrdqty": "volume",
            "total_traded_quantity": "volume",
            "total_traded_qty": "volume",
            "volume": "volume",
        }

        if n in aliases:
            rename[col] = aliases[n]

    df = df.rename(columns=rename)

    missing = [
        c for c in EXPECTED_COLUMNS
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Received columns: {list(df.columns)}"
        )

    df = df[EXPECTED_COLUMNS].copy()

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce",
    ).dt.date

    df["symbol"] = (
        df["symbol"]
        .astype("string")
        .str.strip()
    )

    for col in [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

    df = df.dropna(
        subset=["date", "symbol"]
    )

    if STOCKS is not None:
        df = df[
            df["symbol"].isin(STOCKS)
        ]

    df = df.drop_duplicates(
        subset=["date", "symbol"],
        keep="last",
    )

    return df


def read_nse_zip(content: bytes) -> pd.DataFrame:
    with zipfile.ZipFile(
        io.BytesIO(content)
    ) as z:

        csv_names = [
            name
            for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        if not csv_names:
            raise ValueError(
                "ZIP archive contains no CSV."
            )

        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f)

    return normalize_columns(df)

# ============================================================
# 7. STOCK VALIDATION + ATOMIC PARQUET WRITE
# ============================================================

def validate_dataframe(
    df: pd.DataFrame,
    expected_day: date,
) -> tuple[bool, str]:

    if df.empty:
        return False, "empty_dataframe"

    missing = [
        c for c in EXPECTED_COLUMNS
        if c not in df.columns
    ]

    if missing:
        return False, f"missing_columns:{missing}"

    if df["date"].isna().any():
        return False, "null_dates"

    if not (
        df["date"] == expected_day
    ).all():
        return False, "wrong_date"

    if (
        df["symbol"].isna().any()
        or (
            df["symbol"]
            .astype(str)
            .str.len()
            == 0
        ).any()
    ):
        return False, "invalid_symbols"

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]

    if df[numeric_cols].isna().all(axis=1).any():
        return False, "rows_with_all_numeric_values_null"

    return True, "ok"


def validate_parquet(
    path: Path,
    expected_day: date,
) -> tuple[bool, str, int]:

    if not path.exists():
        return False, "missing_file", 0

    try:
        table = pq.read_table(path)
        df = table.to_pandas()

        df = normalize_columns(df)

        ok, reason = validate_dataframe(
            df,
            expected_day,
        )

        return ok, reason, len(df)

    except Exception as exc:

        return (
            False,
            f"parquet_read_error:{type(exc).__name__}:{exc}",
            0,
        )


def atomic_write_parquet(
    df: pd.DataFrame,
    path: Path,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp_path = path.with_suffix(
        ".parquet.tmp"
    )

    tmp_path.unlink(
        missing_ok=True
    )

    table = pa.Table.from_pandas(
        df,
        preserve_index=False,
    )

    pq.write_table(
        table,
        tmp_path,
        compression="snappy",
    )

    ok, reason, rows = validate_parquet(
        tmp_path,
        df["date"].iloc[0],
    )

    if not ok:
        tmp_path.unlink(
            missing_ok=True
        )

        raise ValueError(
            f"Post-write validation failed: {reason}"
        )

    tmp_path.replace(path)

    return rows

# ============================================================
# 8. STOCK DOWNLOAD
# ============================================================

def download_stock_day(day: date) -> dict:

    path = parquet_path_for_day(day)
    url = url_for_day(day)

    if path.exists():

        ok, reason, rows = validate_parquet(
            path,
            day,
        )

        if ok:
            return {
                "date": day.isoformat(),
                "status": "already_valid",
                "rows": rows,
                "path": str(path),
                "url": url,
                "message": "Existing Parquet passed validation.",
            }

        if not REPAIR_INVALID_FILES:
            return {
                "date": day.isoformat(),
                "status": "invalid_skipped",
                "rows": rows,
                "path": str(path),
                "url": url,
                "message": reason,
            }

        logger.warning(
            "%s invalid: %s — repairing.",
            day,
            reason,
        )

        path.unlink(
            missing_ok=True
        )

    try:

        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
        )

        if response.status_code == 404:
            return {
                "date": day.isoformat(),
                "status": "not_found",
                "rows": 0,
                "path": str(path),
                "url": url,
                "message": (
                    "NSE archive not found; "
                    "likely holiday/non-trading day."
                ),
            }

        response.raise_for_status()

        content = response.content

        if not content:
            raise ValueError(
                "NSE returned an empty response."
            )

        if KEEP_RAW_ZIPS:
            raw_zip_path_for_day(
                day
            ).write_bytes(content)

        df = read_nse_zip(content)

        ok, reason = validate_dataframe(
            df,
            day,
        )

        if not ok:
            raise ValueError(
                f"Downloaded data failed validation: {reason}"
            )

        rows = atomic_write_parquet(
            df,
            path,
        )

        return {
            "date": day.isoformat(),
            "status": "downloaded",
            "rows": rows,
            "path": str(path),
            "url": url,
            "message": "Downloaded successfully.",
        }

    except Exception as exc:

        return {
            "date": day.isoformat(),
            "status": "error",
            "rows": 0,
            "path": str(path),
            "url": url,
            "message": (
                f"{type(exc).__name__}: {exc}"
            ),
        }

# ============================================================
# 9. STOCK MANIFEST
# ============================================================

MANIFEST_COLUMNS = [
    "date",
    "status",
    "rows",
    "path",
    "url",
    "message",
    "checked_at",
]


def load_manifest() -> pd.DataFrame:

    if not MANIFEST_FILE.exists():
        return pd.DataFrame(
            columns=MANIFEST_COLUMNS
        )

    try:

        manifest = pd.read_csv(
            MANIFEST_FILE
        )

        for col in MANIFEST_COLUMNS:
            if col not in manifest.columns:
                manifest[col] = None

        return manifest[
            MANIFEST_COLUMNS
        ]

    except Exception as exc:

        logger.warning(
            "Could not read manifest: %s",
            exc,
        )

        return pd.DataFrame(
            columns=MANIFEST_COLUMNS
        )


manifest = load_manifest()


def append_manifest(record: dict):

    global manifest

    row = {
        col: record.get(col)
        for col in MANIFEST_COLUMNS
    }

    row["checked_at"] = datetime.now().isoformat(
        timespec="seconds"
    )

    manifest = pd.concat(
        [
            manifest,
            pd.DataFrame([row]),
        ],
        ignore_index=True,
    )

    manifest["date"] = (
        manifest["date"].astype(str)
    )

    manifest = (
        manifest
        .drop_duplicates(
            subset=["date"],
            keep="last",
        )
        .sort_values("date")
    )

    manifest.to_csv(
        MANIFEST_FILE,
        index=False,
    )


print(
    "Existing stock manifest records:",
    len(manifest),
)

# ============================================================
# 10. STOCK SYNC
# ============================================================

candidates = list(
    trading_day_candidates(
        START_DATE,
        END_DATE,
    )
)

existing_valid = 0
existing_invalid = 0
missing = 0

for day in candidates:

    path = parquet_path_for_day(day)

    if not path.exists():
        missing += 1
        continue

    ok, _, _ = validate_parquet(
        path,
        day,
    )

    if ok:
        existing_valid += 1
    else:
        existing_invalid += 1


print("Weekday candidates :", f"{len(candidates):,}")
print("Valid existing     :", f"{existing_valid:,}")
print("Invalid existing   :", f"{existing_invalid:,}")
print("Missing            :", f"{missing:,}")

# ============================================================
# 11. STOCK DOWNLOAD LOOP — MISSING + FAILED ONLY
# ============================================================

run_started = datetime.now()
run_results = []

# ------------------------------------------------------------
# BUILD DOWNLOAD QUEUE FROM FILESYSTEM
# ------------------------------------------------------------
#
# candidates = ALL dates in the requested historical range.
#
# We DO NOT iterate over candidates directly.
#
# Existing Parquet:
#     -> SKIP permanently
#
# Missing Parquet:
#     -> DOWNLOAD / RETRY
#
# This means a stale manifest can NEVER cause an existing file
# to be downloaded again.
# ------------------------------------------------------------

stock_existing = []
stock_missing = []

for day in candidates:

    path = parquet_path_for_day(day)

    if path.exists():
        stock_existing.append(day)
    else:
        stock_missing.append(day)


# ------------------------------------------------------------
# HARD ASSERTION
# ------------------------------------------------------------

existing_in_missing = [
    day
    for day in stock_missing
    if parquet_path_for_day(day).exists()
]

if existing_in_missing:
    raise RuntimeError(
        "ABORT: dates classified as missing now have Parquet files:\n"
        + "\n".join(
            str(parquet_path_for_day(day))
            for day in existing_in_missing[:100]
        )
    )


# ------------------------------------------------------------
# FAILED / ERROR RETRIES
# ------------------------------------------------------------
#
# A failed manifest entry is useful for classification, but the
# filesystem remains authoritative.
#
# If the file is missing, it belongs in the queue regardless of
# whether the previous manifest says:
#
#     downloaded
#     failed
#     error
#     not_found
#     anything else
#
# This repairs stale/inconsistent manifests automatically.
# ------------------------------------------------------------

download_queue = list(stock_missing)


# ------------------------------------------------------------
# FINAL QUEUE ASSERTION
# ------------------------------------------------------------

queue_existing = [
    day
    for day in download_queue
    if parquet_path_for_day(day).exists()
]

if queue_existing:
    raise RuntimeError(
        "ABORT: download queue contains existing Parquet files:\n"
        + "\n".join(
            str(parquet_path_for_day(day))
            for day in queue_existing[:100]
        )
    )

assert len(stock_existing) + len(stock_missing) == len(candidates)
assert len(download_queue) == len(stock_missing)


# ------------------------------------------------------------
# PREFLIGHT SUMMARY
# ------------------------------------------------------------

print("=" * 90)
print("STOCK DOWNLOAD PREFLIGHT")
print("=" * 90)

print(f"Candidate dates      : {len(candidates):,}")
print(f"Existing Parquets    : {len(stock_existing):,}")
print(f"Missing Parquets     : {len(stock_missing):,}")
print(f"Download queue       : {len(download_queue):,}")

print()

if not download_queue:

    print("✓ NOTHING TO DOWNLOAD")
    print("✓ All candidate dates already have Parquet files.")

else:

    print("✓ Preflight passed.")
    print("✓ Existing files will NOT be downloaded.")
    print("✓ Only missing files will be requested.")
    print()


# ------------------------------------------------------------
# DOWNLOAD ONLY MISSING FILES
# ------------------------------------------------------------

for i, day in enumerate(
    tqdm(
        download_queue,
        desc="NSE missing/failed files",
    ),
    1,
):

    # --------------------------------------------------------
    # FINAL SAFETY CHECK BEFORE NETWORK REQUEST
    # --------------------------------------------------------

    path = parquet_path_for_day(day)

    if path.exists():
        raise RuntimeError(
            f"ABORT: Parquet appeared before network request:\n{path}"
        )


    # --------------------------------------------------------
    # DOWNLOAD
    # --------------------------------------------------------

    result = download_stock_day(day)

    run_results.append(result)

    append_manifest(result)


    # --------------------------------------------------------
    # RATE LIMITING
    # --------------------------------------------------------

    if result["status"] in {
        "downloaded",
        "error",
        "failed",
    }:

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(
                SLEEP_BETWEEN_REQUESTS
            )


run_finished = datetime.now()

run_df = pd.DataFrame(
    run_results
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print()
print("=" * 90)
print("STOCK SYNC COMPLETED")
print("=" * 90)

print("Started :", run_started)
print("Finished:", run_finished)
print("Elapsed :", run_finished - run_started)

print()
print(f"Existing / skipped : {len(stock_existing):,}")
print(f"Downloaded / retried: {len(run_results):,}")
# ============================================================
# 12. STOCK SYNC SUMMARY
# ============================================================

if run_df.empty:

    print("No stock dates processed.")

else:

    print("Stock status counts:")

    display(
        run_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(
            name="count"
        )
    )

    downloaded_rows = run_df.loc[
        run_df["status"] == "downloaded",
        "rows",
    ].sum()

    print(
        "Rows downloaded:",
        f"{downloaded_rows:,}",
    )

    errors = run_df[
        run_df["status"] == "error"
    ]

    if not errors.empty:

        print("Stock errors:")

        display(
            errors[
                [
                    "date",
                    "message",
                    "url",
                ]
            ]
        )

# 22. How to Run

### First test

For a quick test, use:



In [ ]:

Run all cells and inspect both stock and NIFTY summaries.

### Full historical sync

Then use:



In [ ]:

The notebook is idempotent:

- Valid stock Parquet → skipped
- Missing stock Parquet → downloaded
- Invalid stock Parquet → repaired
- Valid NIFTY 50 Parquet → skipped
- Missing NIFTY 50 Parquet → downloaded
- Invalid NIFTY 50 Parquet → repaired
- NSE 404 / no index record → recorded as non-trading/missing
- Writes are atomic
- Manifests are updated

### Daily use

You can simply rerun the notebook every day with:



In [ ]:

It will only fetch dates that are missing or invalid.

### Important

The NIFTY 50 files are intentionally kept separate from stock files:



## Patched 2025/2026 NSE download logic

This is the only new/modified functionality. It leaves the original historical sync and NIFTY 50 functionality intact and performs a safe post-2024 repair pass for missing/invalid 2025–2026 stock files.


In [ ]:

# ============================================================
# PATCH: NSE STOCK DOWNLOAD LOGIC FOR 2025 + 2026
# ============================================================
#
# This patch is intentionally appended to the original notebook.
#
# It preserves:
#   * 2011 -> present configuration
#   * existing Parquet files
#   * existing manifest
#   * validation helpers
#   * NIFTY 50 code
#   * DuckDB / reporting code
#
# It only changes the stock download behavior for 2025/2026.
#
# IMPORTANT:
# The original notebook's url_for_day()/udiff_url() remains available.
# We override only the network download function used by a new sync pass.
#
# Existing valid Parquet files are NEVER deleted or overwritten.
# A 404 is NOT immediately classified as a holiday.
# ============================================================

PATCH_START_DATE = date(2025, 1, 1)
PATCH_END_DATE = date.today()

PATCH_DIAGNOSTIC_FILE = DATA_DIR / "nse_2025_2026_diagnostics.csv"

# Reuse the original HTTP session if it exists.
# Otherwise create a compatible one.
if "session" not in globals():
    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/131.0 Safari/537.36"
        ),
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.nseindia.com/",
        "Connection": "keep-alive",
    })


def patched_candidate_urls(day: date):
    """
    Candidate NSE archive URLs for the current CM dissemination formats.

    We deliberately try more than one known naming convention so that a
    changed NSE archive name does not get mistaken for a holiday.
    """
    ymd = day.strftime("%Y%m%d")
    ddmmyy = day.strftime("%d%m%y")

    urls = [
        # CM-UDiFF Common Bhavcopy Final
        (
            "https://nsearchives.nseindia.com/content/cm/"
            f"BhavCopy_NSE_CM_0_0_0_{ymd}_F_0000.csv.zip"
        ),

        # NSE PR Bhavcopy naming
        (
            "https://nsearchives.nseindia.com/content/cm/"
            f"PR{ddmmyy}.zip"
        ),

        # Alternate PR naming used by NSE
        (
            "https://nsearchives.nseindia.com/content/cm/"
            f"nupr{ddmmyy}.zip"
        ),
    ]

    return list(dict.fromkeys(urls))


def patched_parse_nse_content(content: bytes, expected_day: date):
    """
    Parse an NSE ZIP/CSV and validate it against the expected trading date.
    Uses the original normalize_columns()/validate_dataframe() helpers.
    """

    # ZIP
    if content[:2] == b"PK":

        with zipfile.ZipFile(io.BytesIO(content)) as z:

            names = [
                name
                for name in z.namelist()
                if name.lower().endswith(
                    (".csv", ".txt", ".dat")
                )
            ]

            if not names:
                raise ValueError(
                    "ZIP contains no CSV/TXT/DAT file."
                )

            errors = []

            for name in names:

                try:

                    with z.open(name) as f:
                        raw = f.read()

                    df = pd.read_csv(
                        io.BytesIO(raw),
                        low_memory=False,
                    )

                    df = normalize_columns(df)

                    ok, reason = validate_dataframe(
                        df,
                        expected_day,
                    )

                    if ok:
                        return df

                    errors.append(
                        f"{name}: {reason}"
                    )

                except Exception as exc:

                    errors.append(
                        f"{name}: {type(exc).__name__}: {exc}"
                    )

            raise ValueError(
                "No valid NSE OHLCV file in ZIP. "
                + " | ".join(errors[:10])
            )

    # Direct CSV
    df = pd.read_csv(
        io.BytesIO(content),
        low_memory=False,
    )

    df = normalize_columns(df)

    ok, reason = validate_dataframe(
        df,
        expected_day,
    )

    if not ok:
        raise ValueError(reason)

    return df


def patched_download_stock_day(day: date) -> dict:
    """
    Safe post-2024 downloader.

    Existing valid Parquet:
        -> untouched

    HTTP 404:
        -> try every known NSE URL first

    All URLs 404:
        -> probable holiday/non-trading

    Server/network errors:
        -> explicit error status

    Successful download:
        -> validated before atomic replacement
    """

    path = parquet_path_for_day(day)

    # --------------------------------------------------------
    # Existing valid Parquet is authoritative.
    # --------------------------------------------------------

    if path.exists():

        ok, reason, rows = validate_parquet(
            path,
            day,
        )

        if ok:
            return {
                "date": day.isoformat(),
                "status": "already_valid",
                "rows": rows,
                "path": str(path),
                "url": "",
                "message": "Existing Parquet preserved.",
            }

        logger.warning(
            "%s existing Parquet invalid: %s",
            day,
            reason,
        )

        # IMPORTANT:
        # Do NOT delete the existing file.
        # Download into a temporary path and replace only after success.

    attempts = []

    for url in patched_candidate_urls(day):

        try:

            response = session.get(
                url,
                timeout=REQUEST_TIMEOUT,
            )

            status = response.status_code

            # ------------------------------------------------
            # Successful response
            # ------------------------------------------------

            if status == 200:

                if not response.content:

                    attempts.append({
                        "url": url,
                        "status": "empty_response",
                        "http_status": status,
                    })

                    continue

                try:

                    df = patched_parse_nse_content(
                        response.content,
                        day,
                    )

                except Exception as exc:

                    attempts.append({
                        "url": url,
                        "status": "parse_error",
                        "http_status": status,
                        "message": str(exc),
                    })

                    continue

                # ------------------------------------------------
                # Atomic write.
                #
                # Existing valid files have already returned above.
                # Invalid files are replaced only after successful
                # download + validation.
                # ------------------------------------------------

                try:

                    rows = atomic_write_parquet(
                        df,
                        path,
                    )

                except Exception as exc:

                    return {
                        "date": day.isoformat(),
                        "status": "validation_error",
                        "rows": 0,
                        "path": str(path),
                        "url": url,
                        "message": str(exc),
                    }

                return {
                    "date": day.isoformat(),
                    "status": "downloaded",
                    "rows": rows,
                    "path": str(path),
                    "url": url,
                    "message": "Downloaded and validated.",
                }

            # ------------------------------------------------
            # 404 — DO NOT CALL IT A HOLIDAY YET
            # ------------------------------------------------

            if status == 404:

                attempts.append({
                    "url": url,
                    "status": "url_not_found",
                    "http_status": status,
                })

                continue

            # ------------------------------------------------
            # Rate limiting
            # ------------------------------------------------

            if status == 429:

                attempts.append({
                    "url": url,
                    "status": "rate_limited",
                    "http_status": status,
                })

                time.sleep(
                    min(
                        30,
                        2 ** len(attempts),
                    )
                )

                continue

            # ------------------------------------------------
            # NSE server error
            # ------------------------------------------------

            if status >= 500:

                attempts.append({
                    "url": url,
                    "status": "server_error",
                    "http_status": status,
                })

                continue

            # ------------------------------------------------
            # Other HTTP response
            # ------------------------------------------------

            attempts.append({
                "url": url,
                "status": "http_error",
                "http_status": status,
                "message": response.text[:500],
            })

        except requests.RequestException as exc:

            attempts.append({
                "url": url,
                "status": "network_error",
                "http_status": "",
                "message": str(exc),
            })

    statuses = [
        x["status"]
        for x in attempts
    ]

    http_statuses = [
        x.get("http_status")
        for x in attempts
        if x.get("http_status")
    ]

    # --------------------------------------------------------
    # All known archive URLs returned 404.
    #
    # This is a PROBABLE non-trading day, not a generic
    # "404 means holiday" rule.
    # --------------------------------------------------------

    if (
        attempts
        and all(
            status == "url_not_found"
            for status in statuses
        )
    ):

        return {
            "date": day.isoformat(),
            "status": "holiday_or_nontrading",
            "rows": 0,
            "path": str(path),
            "url": ";".join(
                x["url"]
                for x in attempts
            ),
            "message": (
                "All known NSE archive URLs returned 404. "
                "Probable non-trading/holiday."
            ),
        }

    if "server_error" in statuses:

        return {
            "date": day.isoformat(),
            "status": "server_error",
            "rows": 0,
            "path": str(path),
            "url": ";".join(
                x["url"]
                for x in attempts
            ),
            "message": json.dumps(
                attempts
            )[:4000],
        }

    if "network_error" in statuses:

        return {
            "date": day.isoformat(),
            "status": "network_error",
            "rows": 0,
            "path": str(path),
            "url": ";".join(
                x["url"]
                for x in attempts
            ),
            "message": json.dumps(
                attempts
            )[:4000],
        }

    if "parse_error" in statuses:

        return {
            "date": day.isoformat(),
            "status": "parse_error",
            "rows": 0,
            "path": str(path),
            "url": ";".join(
                x["url"]
                for x in attempts
            ),
            "message": json.dumps(
                attempts
            )[:4000],
        }

    return {
        "date": day.isoformat(),
        "status": "download_error",
        "rows": 0,
        "path": str(path),
        "url": ";".join(
            x["url"]
            for x in attempts
        ),
        "message": json.dumps(
            attempts
        )[:4000],
    }


# ============================================================
# PATCHED 2025/2026 SYNC
# ============================================================

patch_candidates = list(
    trading_day_candidates(
        PATCH_START_DATE,
        PATCH_END_DATE,
    )
)

patch_queue = []

for day in patch_candidates:

    path = parquet_path_for_day(day)

    if path.exists():

        ok, _, _ = validate_parquet(
            path,
            day,
        )

        if ok:
            continue

    patch_queue.append(day)


print("=" * 90)
print("PATCHED NSE 2025 + 2026 STOCK SYNC")
print("=" * 90)

print(
    "Range:",
    PATCH_START_DATE,
    "to",
    PATCH_END_DATE,
)

print(
    "Weekday candidates:",
    f"{len(patch_candidates):,}",
)

print(
    "Missing / invalid:",
    f"{len(patch_queue):,}",
)

print(
    "Existing valid Parquets will NOT be downloaded."
)

print()


patch_results = []

for day in tqdm(
    patch_queue,
    desc="NSE 2025/2026 patched sync",
):

    result = patched_download_stock_day(day)

    patch_results.append(
        result
    )

    # Preserve the existing manifest.
    if "append_manifest" in globals():

        append_manifest(result)

    if SLEEP_BETWEEN_REQUESTS:
        time.sleep(
            SLEEP_BETWEEN_REQUESTS
        )


patch_df = pd.DataFrame(
    patch_results
)

if not patch_df.empty:

    patch_df.to_csv(
        PATCH_DIAGNOSTIC_FILE,
        index=False,
    )


print()
print("=" * 90)
print("PATCHED SYNC COMPLETE")
print("=" * 90)

if patch_df.empty:

    print(
        "No 2025/2026 files required downloading."
    )

else:

    display(
        patch_df[
            "status"
        ]
        .value_counts()
        .rename_axis(
            "status"
        )
        .reset_index(
            name="count"
        )
    )

    print()

    print(
        "Rows downloaded:",
        f"{patch_df.loc[patch_df['status'] == 'downloaded', 'rows'].sum():,}",
    )

    problems = patch_df[
        patch_df["status"].isin([
            "server_error",
            "network_error",
            "parse_error",
            "validation_error",
            "download_error",
        ])
    ]

    if not problems.empty:

        print(
            "PROBLEMS:"
        )

        display(
            problems[
                [
                    "date",
                    "status",
                    "message",
                    "url",
                ]
            ]
            .head(100)
        )

    print()
    print(
        "Diagnostics:",
        PATCH_DIAGNOSTIC_FILE,
    )
